In [7]:
import pickle
import cv2
import mediapipe as mp
from collections import deque
import time

# Load model
model_dict = pickle.load(open('./model.p', 'rb'))
model = model_dict['model']

# MediaPipe
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    min_detection_confidence=0.5,
    max_num_hands=1
)

# Feature extraction
def extract_hand_features(hand_landmarks):
    x_, y_ = [], []

    for lm in hand_landmarks.landmark:
        x_.append(lm.x)
        y_.append(lm.y)

    data_aux = []
    for lm in hand_landmarks.landmark:
        data_aux.append(lm.x - min(x_))
        data_aux.append(lm.y - min(y_))

    return data_aux


# 🔹 Sentence system
buffer = deque(maxlen=10)
sentence = ""
prev_char = ""
last_added_time = 0
cooldown = 1.0


cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    frame = cv2.flip(frame, 1)

    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(img_rgb)

    display_char = ""

    if results.multi_hand_landmarks:
        hand_landmarks = results.multi_hand_landmarks[0]

        features = extract_hand_features(hand_landmarks)
        prediction = model.predict([features])[0]

        buffer.append(prediction)

        mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

        if len(buffer) == buffer.maxlen:
            stable_char = max(set(buffer), key=buffer.count)
            display_char = stable_char

            current_time = time.time()

            if stable_char != prev_char and (current_time - last_added_time) > cooldown:
                if stable_char == "space":
                    sentence += " "
                else:
                    sentence += stable_char

                prev_char = stable_char
                last_added_time = current_time

    # 🖥️ Display
    cv2.putText(frame, f'Current: {display_char}',
                (50, 50), cv2.FONT_HERSHEY_SIMPLEX,
                1, (0, 255, 0), 2)

    cv2.putText(frame, f'Sentence: {sentence}',
                (50, 100), cv2.FONT_HERSHEY_SIMPLEX,
                1, (255, 255, 0), 2)

    cv2.putText(frame, 'Controls: Q=Quit | C=Clear | B=Backspace | W=Word Delete',
                (50, 150), cv2.FONT_HERSHEY_SIMPLEX,
                0.6, (200, 200, 200), 2)

    cv2.imshow('ISL Recognition', frame)

    key = cv2.waitKey(1) & 0xFF

    # ❌ Quit
    if key == ord('q'):
        break

    # 🧽 Clear full sentence
    elif key == ord('c'):
        sentence = ""
        prev_char = ""

    # ⌫ Delete last character
    elif key == ord('b'):
        sentence = sentence[:-1]

    # 🧹 Delete last word
    elif key == ord('w'):
        sentence = sentence.rstrip()  # remove trailing space
        sentence = " ".join(sentence.split(" ")[:-1])

cap.release()
cv2.destroyAllWindows()